In [1]:
!pip install langgraph langchain

  Using cached langgraph-1.0.10-py3-none-any.whl (160 kB)
  Using cached langchain-1.2.10-py3-none-any.whl (111 kB)
  Using cached langchain_core-1.2.17-py3-none-any.whl (502 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl (50 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl (35 kB)
  Using cached langgraph_sdk-0.3.9-py3-none-any.whl (90 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached xxhash-3.6.0-cp311-cp311-win_amd64.whl (31 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
     ------------------------------------ 347.4/347.4 kB 830.3 kB/s eta 0:00:00
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl (158 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl (187 kB)
  Using cached ormsgpack-1.12.2-cp311-cp311-win_amd64.whl (117 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached orjson-3.11.7-cp311-cp311-win_amd64.whl (124 kB)
  U


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip install python-dotenv langchain-groq langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 1.6 MB/s  0:00:00
Using cached marshmallow-3.26.2-py3-n

In [14]:
from typing import List, Optional
from pydantic import BaseModel, Field
import os
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import requests
from typing import List, Dict, Any, Optional,Tuple
from langchain.tools import tool
import http.client
import json
import time
from urllib.parse import quote
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain_core.output_parsers import JsonOutputParser
from datetime import datetime
import datetime



In [5]:
class TripPlan(BaseModel):
  """Data model for the user's trip plan. Now stores raw user inputs as strings."""

  destination: Optional[str] = Field(None, description="Destination city/country")
  budget: Optional[str] = Field(
    None, description="Budget for the trip (raw user input)"
  )
  native_currency: Optional[str] = Field(None, description="User's native currency")
  days: Optional[str] = Field(
    None, description="Duration of the trip in days (raw user input)"
  )
  group_size: Optional[str] = Field(
    "1", description="Number of people traveling (raw user input)"
  )
  activity_preferences: Optional[str] = Field(
    None,
    description="Preferences like adventure, culture, relaxation (raw user input)",
  )
  accommodation_type: Optional[str] = Field(
    None,
    description="Preferred accommodation type (e.g., hotel, hostel, airbnb)",
  )
  dietary_restrictions: Optional[str] = Field(
    None, description="Any dietary restrictions (raw user input)"
  )
  transportation_preferences: Optional[str] = Field(
    None, description="Preferred mode of transport (raw user input)"
  )


class QueryAnalysisResult(TripPlan):
  """Data model for the output of the QueryAnalyzer."""

  missing_fields: List[str] = Field(
    default_factory=list,
    description="List of required fields that are missing from the user query",
  )

# default_factory=list means that if missing_fields is not provided
# when creating a QueryAnalysisResult, it will default to [] (a new, empty list).
# This avoids potential issues with all instances sharing the same list.

class WorkflowState(TripPlan):
    """State for the workflow, including conversation history and all planning fields."""
    messages: list = Field(default_factory=list)
    hotels: Optional[list] = None
    attractions: Optional[str] = None
    weather: Optional[str] = None
    itinerary: Optional[dict] = None
    summary: Optional[dict] = None
    currency_rates: Optional[str] = None
    missing_fields: Optional[list] = None
    calculator_result: Optional[str] = None
    prompt: Optional[str] = None

class HotelInfo(BaseModel):
    """Minimal hotel info for workflow use."""
    name: str
    price_per_night: float
    review_count: int
    rating: Optional[float] = None
    url: Optional[str] = None



In [7]:
!pip install python-dotenv

In [12]:
load_dotenv()

# Groq API
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Tavily Search
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

tavily_search = TavilySearchResults(max_results=5)

# Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

C:\Users\Divyanshu\AppData\Local\Temp\ipykernel_18852\3943215143.py:9: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=5)
